# Steriflow MPI1 Data Exploration

Notebook goals:
- Read all `.txt` files from the MPI1 export folder.
- Sort files by modification date.
- Ignore lines 1 to 10.
- Use line 11 as headers and line 12 as active/inactive flags.
- Keep datetime found in square brackets in filename as `batch_datetime`.
- Merge files with and without `_II` suffix for the same batch.
- Build a combined dataframe and plot time series for exploration.


In [1]:
# Setup
from __future__ import annotations

import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 200)

DATA_DIR = Path(r"D:\\Cascadya\\Cascadya - Documents\\08. COMPTE CLIENT\\Inariz_Lamballe\\2. Données sous NDA\\données Steriflow\\MPI1_csv")
DATA_DIR


WindowsPath('D:/Cascadya/Cascadya - Documents/08. COMPTE CLIENT/Inariz_Lamballe/2. Données sous NDA/données Steriflow/MPI1_csv')

## Helpers for parsing and merging

Rules implemented exactly as requested:
- sort files by modification datetime,
- line 11 = headers,
- line 12 = active/inactive flags,
- load all columns,
- merge base and `_II` companion files for the same batch datetime.


In [ ]:
FILENAME_DT_RE = re.compile(r"\[(\d{2}_\d{2}_\d{4}_\d{2}_\d{2}_\d{2})\]")


def parse_batch_datetime(file_path: Path) -> pd.Timestamp | pd.NaT:
    match = FILENAME_DT_RE.search(file_path.name)
    if not match:
        return pd.NaT
    return pd.to_datetime(match.group(1), format="%d_%m_%Y_%H_%M_%S", errors="coerce")


def normalize_batch_key(file_path: Path) -> str:
    stem = file_path.stem
    if stem.endswith("_II"):
        stem = stem[:-3]
    return stem


def suffix_type(file_path: Path) -> str:
    return "II" if file_path.stem.endswith("_II") else "base"


def make_unique_headers(raw_headers: list[str]) -> list[str]:
    headers: list[str] = []
    seen: dict[str, int] = {}
    for idx, h in enumerate(raw_headers):
        clean = h.strip()
        if clean == "":
            clean = f"_unnamed_{idx}"
        count = seen.get(clean, 0)
        seen[clean] = count + 1
        headers.append(clean if count == 0 else f"{clean}_{count + 1}")
    return headers


def read_steriflow_txt(file_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    lines = file_path.read_text(encoding="utf-16", errors="replace").splitlines()
    if len(lines) < 12:
        raise ValueError(f"File has fewer than 12 lines: {file_path}")

    header_line = lines[10]
    active_line = lines[11]
    data_lines = lines[12:]

    raw_headers = [c.strip() for c in header_line.split(";")]
    active_flags = [c.strip() for c in active_line.split(";")]
    headers = make_unique_headers(raw_headers)

    n_cols = len(headers)

    rows: list[list[str]] = []
    for line in data_lines:
        if not line.strip():
            continue
        values = line.split(";")
        if len(values) < n_cols:
            values += [""] * (n_cols - len(values))
        elif len(values) > n_cols:
            values = values[:n_cols]
        rows.append(values)

    df = pd.DataFrame(rows, columns=headers)

    df = df[df.columns.drop(list(df.filter(regex='unnamed')))]

    for col in df.columns:
        df[col] = df[col].astype(str).str.strip()

    dt = parse_batch_datetime(file_path)
    df["batch_datetime"] = dt
    df["batch_key"] = normalize_batch_key(file_path)
    df["file_variant"] = suffix_type(file_path)
    df["source_file"] = file_path.name
    df["file_mtime"] = pd.to_datetime(file_path.stat().st_mtime, unit="s")



    active_flags += [""] * (n_cols - len(active_flags))
    activity = pd.DataFrame(
        {
            "column": headers,
            "active_flag": active_flags[:n_cols],
            "source_file": file_path.name,
            "file_variant": suffix_type(file_path),
            "batch_datetime": dt,
            "batch_key": normalize_batch_key(file_path),
        }
    )

    return df, activity


def to_numeric_best_effort(df: pd.DataFrame, skip: set[str]) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        if col in skip:
            continue
        if out[col].dtype == object:
            cleaned = out[col].str.replace(",", ".", regex=False)
            num = pd.to_numeric(cleaned, errors="coerce")
            if num.notna().sum() > 0:
                out[col] = num
    return out


def merge_batch_variants(batch_df: pd.DataFrame) -> pd.DataFrame:
    base = batch_df[batch_df["file_variant"] == "base"].copy()
    ii = batch_df[batch_df["file_variant"] == "II"].copy()

    merge_keys = [k for k in ["Instant", "Phase", "batch_datetime", "batch_key"] if k in batch_df.columns]

    if base.empty:
        return ii
    if ii.empty:
        return base

    drop_meta = {"file_variant", "source_file", "file_mtime"}
    base_data = base.drop(columns=[c for c in drop_meta if c in base.columns])
    ii_data = ii.drop(columns=[c for c in drop_meta if c in ii.columns])

    merged = base_data.merge(ii_data, on=merge_keys, how="outer", suffixes=("_base", "_ii"))
    merged["source_file"] = "; ".join(sorted(batch_df["source_file"].dropna().unique()))
    merged["file_mtime"] = batch_df["file_mtime"].max()
    merged["file_variant"] = "merged"
    return merged


def make_time_axis(df: pd.DataFrame) -> pd.Series:
    if "Instant" not in df.columns:
        return pd.Series(df.index, index=df.index, name="row_index")

    instant = df["Instant"].astype(str)
    td = pd.to_timedelta(instant, errors="coerce")
    if td.notna().sum() > 0:
        return td.dt.total_seconds().rename("instant_seconds")

    dt = pd.to_datetime(instant, errors="coerce", dayfirst=True)
    if dt.notna().sum() > 0:
        return dt.rename("instant_datetime")

    return pd.Series(df.index, index=df.index, name="row_index")


## Load all files and combine base + `_II`

This cell outputs:
- `df_all_raw`: concatenation of every parsed file,
- `df_all`: merged per batch key/date with numeric conversion,
- `activity_flags`: line-12 active/inactive info for every source file.


In [ ]:
txt_files = sorted(DATA_DIR.glob("*.txt"), key=lambda p: p.stat().st_mtime)
if not txt_files:
    raise FileNotFoundError(f"No .txt files found in {DATA_DIR}")

parsed_frames: list[pd.DataFrame] = []
activity_frames: list[pd.DataFrame] = []

for path in txt_files:
    df_file, df_activity = read_steriflow_txt(path)
    parsed_frames.append(df_file)
    activity_frames.append(df_activity)

df_all_raw = pd.concat(parsed_frames, ignore_index=True)
activity_flags = pd.concat(activity_frames, ignore_index=True)

merged_batches: list[pd.DataFrame] = []
for (_, _), batch in df_all_raw.groupby(["batch_key", "batch_datetime"], dropna=False):
    merged_batches.append(merge_batch_variants(batch))

df_all = pd.concat(merged_batches, ignore_index=True)

meta_cols = {
    "Instant",
    "Phase",
    "batch_datetime",
    "batch_key",
    "file_variant",
    "source_file",
    "file_mtime",
}
df_all = to_numeric_best_effort(df_all, skip=meta_cols)


In [ ]:
print(parsed_frames[0].loc[0]['Déflexion (mm)'])
print(parsed_frames[0].loc[0]['batch_datetime'])
print(parsed_frames[0].loc[0]['batch_key'])
print(parsed_frames[0].loc[0]['file_variant'])
print(parsed_frames[0].loc[0]['source_file'])
print(parsed_frames[0].loc[0]['file_mtime'])


print(parsed_frames[1].loc[0]['P. Diff. Ppe (Bar)'])
print(parsed_frames[1].loc[0]['batch_datetime']) # le seul qu'on gare après le merge
print(parsed_frames[1].loc[0]['batch_key']) # on commente, mais on utilie pas tnat qu'on ne sais pas ce que ça signifie
print(parsed_frames[1].loc[0]['file_variant']) 
print(parsed_frames[1].loc[0]['source_file']) #on garde avant le merge puis on jete
print(parsed_frames[1].loc[0]['file_mtime'])

0.00
2026-03-02 18:58:01
l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_58_01]
base
l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_58_01].txt
2026-03-02 19:49:29.134594917
-0.02
2026-03-02 18:58:01
l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_58_01]
II
l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_58_01]_II.txt
2026-03-02 19:49:32.513788223


In [ ]:

df_all_raw = pd.concat(parsed_frames, ignore_index=True)
activity_flags = pd.concat(activity_frames, ignore_index=True)

merged_batches: list[pd.DataFrame] = []
for (_, _), batch in df_all_raw.groupby(["batch_key", "batch_datetime"], dropna=False):
    merged_batches.append(merge_batch_variants(batch))

df_all = pd.concat(merged_batches, ignore_index=True)

meta_cols = {
    "Instant",
    "Phase",
    "batch_datetime",
    "batch_key",
    "file_variant",
    "source_file",
    "file_mtime",
}
df_all = to_numeric_best_effort(df_all, skip=meta_cols)


In [18]:
df_all = df_all.sort_values(["batch_datetime", "file_mtime", "Instant"], na_position="last").reset_index(drop=True)

print(f"Files read: {len(txt_files)}")
print(f"Rows in raw concatenation: {len(df_all_raw):,}")
print(f"Rows after base/_II merge: {len(df_all):,}")
display(df_all.head(5))
display(activity_flags.head(10))


Files read: 378
Rows in raw concatenation: 25,794
Rows after base/_II merge: 12,897


,_unnamed_0_base,Instant,Phase,Consigne (°C)_base,Mesure Regul. (°C)_base,Consigne (Bar)_base,Mesure (Bar)_base,T. Pt Froid (°C)_base,T. Pt Froid 2 (°C)_base,Niveau (cm)_base,T. Ballast (°C)_base,Vitesse (Tr/min)_base,Débit (m3/h)_base,Déflexion (mm)_base,_unnamed_14_base,_unnamed_15_base,_unnamed_16_base,_unnamed_17_base,_unnamed_18_base,_unnamed_19_base,_unnamed_20_base,_unnamed_21_base,batch_datetime,batch_key,Mesure 1 Regul. (°C)_base,Mesure 2 Regul. (°C)_base,Mesure 1 (Bar)_base,Mesure 2 (Bar)_base,D. Pied Av. (mm)_base,D. Pied Ar. (mm)_base,Vol. Eau (l/h)_base,Puiss. Elec. (kW)_base,P. Diff. Ppe (Bar)_base,_unnamed_12_base,_unnamed_13_base,_unnamed_22_base,_unnamed_23_base,_unnamed_0_ii,Consigne (°C)_ii,Mesure Regul. (°C)_ii,Consigne (Bar)_ii,Mesure (Bar)_ii,T. Pt Froid (°C)_ii,T. Pt Froid 2 (°C)_ii,Niveau (cm)_ii,T. Ballast (°C)_ii,Vitesse (Tr/min)_ii,Débit (m3/h)_ii,Déflexion (mm)_ii,_unnamed_14_ii,_unnamed_15_ii,_unnamed_16_ii,_unnamed_17_ii,_unnamed_18_ii,_unnamed_19_ii,_unnamed_20_ii,_unnamed_21_ii,Mesure 1 Regul. (°C)_ii,Mesure 2 Regul. (°C)_ii,Mesure 1 (Bar)_ii,Mesure 2 (Bar)_ii,D. Pied Av. (mm)_ii,D. Pied Ar. (mm)_ii,Vol. Eau (l/h)_ii,Puiss. Elec. (kW)_ii,P. Diff. Ppe (Bar)_ii,_unnamed_12_ii,_unnamed_13_ii,_unnamed_22_ii,_unnamed_23_ii,source_file,file_mtime,file_variant
0,,19:00:01,0,0.00,33.03,0.00,0.01,25.34,0.0,21.88,0.0,0.00,0.11,0.0,,1,0,1,1,1,0,,2026-03-02 18:58:01,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,1.0,0.0,0.0,0.0,0.0,0.0,1.0,33.03,33.07,0.01,0.0,0.0,0.0,0.0,0.0,-0.02,,,,,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...,2026-03-02 19:49:32.513788223,merged
1,,19:02:01,0,0.00,52.22,0.00,0.06,57.27,0.0,17.91,0.0,0.00,311.82,0.0,,1,0,1,1,1,0,,2026-03-02 18:58:01,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,52.22,51.16,0.06,0.0,0.0,0.0,0.0,0.0,0.99,,,,,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...,2026-03-02 19:49:32.513788223,merged
2,,19:04:01,1,54.95,54.02,0.04,0.04,51.77,0.0,19.32,0.0,0.98,395.79,0.0,,1,0,1,1,1,0,,2026-03-02 18:58:01,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,54.02,53.96,0.04,0.0,0.0,0.0,0.0,0.0,1.03,,,,,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...,2026-03-02 19:49:32.513788223,merged
3,,19:06:01,1,64.22,63.62,0.12,0.14,60.88,0.0,19.52,0.0,0.98,388.22,0.0,,1,0,1,1,1,0,,2026-03-02 18:58:01,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63.62,63.65,0.14,0.0,0.0,0.0,0.0,0.0,0.99,,,,,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...,2026-03-02 19:49:32.513788223,merged
4,,19:08:01,1,73.49,71.81,0.18,0.19,69.78,0.0,20.53,0.0,0.98,392.98,0.0,,1,0,1,1,1,0,,2026-03-02 18:58:01,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,71.81,71.75,0.19,0.0,0.0,0.0,0.0,0.0,1.03,,,,,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...,2026-03-02 19:49:32.513788223,merged


,column,active_flag,source_file,file_variant,batch_datetime,batch_key
0,_unnamed_0,,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...,base,2026-03-02 18:58:01,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...
1,Instant,,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...,base,2026-03-02 18:58:01,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...
2,Phase,,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...,base,2026-03-02 18:58:01,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...
3,Consigne (°C),,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...,base,2026-03-02 18:58:01,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...
4,Mesure Regul. (°C),,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...,base,2026-03-02 18:58:01,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...
5,Consigne (Bar),,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...,base,2026-03-02 18:58:01,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...
6,Mesure (Bar),,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...,base,2026-03-02 18:58:01,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...
7,T. Pt Froid (°C),,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...,base,2026-03-02 18:58:01,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...
8,T. Pt Froid 2 (°C),,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...,base,2026-03-02 18:58:01,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...
9,Niveau (cm),[actif],l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...,base,2026-03-02 18:58:01,l6-028-002-7_5_+_10_min_123°C_[02_03_2026_18_5...


In [25]:
# df_all.columns
df_all["Consigne (°C)_ii"].unique()

# batch_datetime, batch_key

array([nan], dtype=object)

## Quick data quality checks

In [ ]:
print("Columns:", len(df_all.columns))
print("Distinct batches:", df_all["batch_key"].nunique(dropna=False))
print("Batch datetime min/max:")
display(df_all["batch_datetime"].agg(["min", "max"]))

num_cols = [c for c in df_all.columns if pd.api.types.is_numeric_dtype(df_all[c])]
print(f"Numeric columns detected: {len(num_cols)}")
display(pd.Series(num_cols).head(20))

activity_summary = (
    activity_flags.assign(active_flag=lambda d: d["active_flag"].str.lower())
    .groupby(["file_variant", "active_flag"], dropna=False)
    .size()
    .rename("count")
    .reset_index()
)
display(activity_summary)


## Time series exploration plots

The notebook automatically picks the most populated numeric columns and plots them in small multiples.


In [ ]:
time_axis = make_time_axis(df_all)
plot_df = df_all.copy()
plot_df[time_axis.name] = time_axis

numeric_cols = [
    c for c in plot_df.columns
    if pd.api.types.is_numeric_dtype(plot_df[c]) and c not in {time_axis.name}
]

if not numeric_cols:
    raise ValueError("No numeric columns found to plot.")

ranked = sorted(numeric_cols, key=lambda c: plot_df[c].notna().sum(), reverse=True)
top_cols = ranked[:12]

n = len(top_cols)
ncols = 3
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(18, 4 * nrows), sharex=True)
axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

for i, col in enumerate(top_cols):
    ax = axes[i]
    ax.plot(plot_df[time_axis.name], plot_df[col], linewidth=1)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=25)

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

axes[0].set_xlabel(time_axis.name)
plt.tight_layout()
plt.show()


## Optional: focus on one batch

Use this section to inspect one sterilization run in detail.


In [ ]:
selected_batch = df_all["batch_key"].dropna().iloc[0] if not df_all.empty else None
batch_df = df_all[df_all["batch_key"] == selected_batch].copy() if selected_batch is not None else pd.DataFrame()

if batch_df.empty:
    print("No batch available.")
else:
    t = make_time_axis(batch_df)
    batch_df[t.name] = t
    batch_num = [c for c in batch_df.columns if pd.api.types.is_numeric_dtype(batch_df[c]) and c != t.name]
    keep = batch_num[:6]
    if not keep:
        print("Selected batch has no numeric columns to plot.")
    else:
        ax = batch_df.plot(x=t.name, y=keep, figsize=(14, 6), linewidth=1)
        ax.set_title(f"Selected batch: {selected_batch}")
        ax.set_xlabel(t.name)
        plt.xticks(rotation=25)
        plt.tight_layout()
        plt.show()

    display(batch_df.head(10))


## Notes

- The `activity_flags` dataframe preserves line-12 active/inactive state per file and column.
- `df_all` keeps `batch_datetime` extracted from `[dd_mm_YYYY_HH_MM_SS]` in filenames.
- Base and `_II` files are merged by `Instant`, `Phase`, `batch_key`, and `batch_datetime`.
